In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [8]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()

    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()

    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [9]:
"""
Determine investment windows where the available balance exceeds a threshold and track the duration of these windows.
df: DataFrame with 'TransactionDate' and 'Available' columns
threshold = 1,000,000 - minimum balance to consider
Returns the investment windows as a DataFrame with columns:
    'start_date': Start date of the investment window
    'end_date': End date of the investment window (None if ongoing)
    'amount': The balance amount during the window
    'duration': Duration of the window in days
"""
# Function to detect window intervals
def find_window_intervals(df, threshold=1_000_000):
    df = df.sort_values('TransactionDate').reset_index(drop=True)
    intervals = []

    for i in range(len(df)):
        base_row = df.iloc[i]
        base_amount = base_row['Available']
        if base_amount < threshold:
            continue

        start_date = base_row['TransactionDate']
        duration = 1
        end_date = None

        for j in range(i + 1, len(df)):
            if df.iloc[j]['Available'] >= base_amount:
                duration += 1
            else:
                end_date = df.iloc[j]['TransactionDate']
                break
        else:
            end_date = df.iloc[-1]['TransactionDate']

        if not intervals or intervals[-1]['start_date'] != start_date:
            intervals.append({
                'start_date': start_date,
                'end_date': end_date,
                'amount': base_amount,
                'duration': duration
            })

    # Create DataFrame
    df_intervals = pd.DataFrame(intervals)

    # Format columns
    df_intervals['start_date'] = pd.to_datetime(df_intervals['start_date']).dt.date
    df_intervals['end_date'] = pd.to_datetime(df_intervals['end_date']).dt.date
    df_intervals['amount'] = df_intervals['amount'].round(2)
    df_intervals['change'] = (df_intervals['amount'] - df_intervals['amount'].shift(1).fillna(0)).round(2)

    return df_intervals


In [10]:
# Step 2: Keep Only the Max Amount Interval for Each Overlapping Period
def drop_overlapping_lower_intervals(df):
    # Group by start_date and end_date, then get the row with maximum available amount
    result = df.loc[df.groupby(['start_date', 'end_date'])['amount'].idxmax()]
    return result

In [11]:
# Step 3: Update Changed Amounts Based on Prior Date from the Original Data
""""
- Iterative lookback: For each interval, the code now loops backward through dates until it finds a balance that's different (from the current amount)
- Dictionary lookup: Uses a dictionary for O(1) date lookups instead of repeated merges
- Safety limit: Includes a max_lookback_days parameter (set to 365) to prevent infinite loops if all historical balances are identical
- Comparison logic: Compares rounded values to ensure we're finding genuinely different amounts (accounting for floating point precision)
- Fallback: If no different amount is found after looking back, it defaults to 0
"""

def update_changed_amounts(df_intervals, df_data):
    df_intervals['start_date'] = pd.to_datetime(df_intervals['start_date'])
    df_data['TransactionDate'] = pd.to_datetime(df_data['TransactionDate'])

    # Create a dictionary for faster lookups of amounts by date
    amount_by_date = df_data.set_index('TransactionDate')['Available'].to_dict()

    previous_amounts = []

    for start_date, current_amount in zip(df_intervals['start_date'], df_intervals['amount']):
        # Start looking from the day before
        lookback_date = start_date - pd.Timedelta(days=1)
        previous_amount = None
        max_lookback_days = 365  # Prevent infinite loops
        days_searched = 0

        while days_searched < max_lookback_days:
            if lookback_date in amount_by_date:
                candidate_amount = amount_by_date[lookback_date]
                # Check if the amount is different from current amount
                if round(candidate_amount, 2) != round(current_amount, 2):
                    previous_amount = candidate_amount
                    break
                else:
                    pass
            # Move back one more day
            lookback_date -= pd.Timedelta(days=1)
            days_searched += 1

        # If no different amount found, use 0 (handles first record case)
        if previous_amount is None:
            previous_amount = 0

        previous_amounts.append(previous_amount)

    df_intervals['previous_amount'] = previous_amounts
    df_intervals['change'] = (df_intervals['amount'] - df_intervals['previous_amount']).round(2)

    # Replace -0.0 with 0.0
    df_intervals['change'] = df_intervals['change'].apply(lambda x: 0.0 if x == 0 else x)

    return df_intervals

In [12]:
# Main execution
    # Note: In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 5
# Load and print the data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-12-31')

# with pd.option_context('display.max_columns', None,
#                        'display.max_rows', None,
#                        'display.width', None,
#                        'display.expand_frame_repr', False):
#     pd.options.display.float_format = '${:,.2f}'.format
# display(data.head(30))

# data.info()

all_intervals = find_window_intervals(data).sort_values(['start_date'], ascending=[True])
intervals = drop_overlapping_lower_intervals(all_intervals)
final_intervals = update_changed_amounts(intervals, data)

# Display
print("\nInvestment windows:", len(intervals))
with pd.option_context('display.max_columns', None,
                       'display.max_rows', 100,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
display(final_intervals.head(25))


Investment windows: 103


,start_date,end_date,amount,duration,change,previous_amount
0,2025-09-04,2025-09-24,"$4,642,791.81",20,"$4,642,791.81",$0.00
1,2025-09-05,2025-09-10,"$10,282,791.81",5,"$5,640,000.00","$4,642,791.81"
2,2025-09-06,2025-09-10,"$10,282,791.81",4,"$5,640,000.00","$4,642,791.81"
3,2025-09-07,2025-09-10,"$10,282,791.81",3,"$5,640,000.00","$4,642,791.81"
4,2025-09-08,2025-09-10,"$23,637,591.81",2,"$13,354,800.00","$10,282,791.81"
5,2025-09-09,2025-09-10,"$28,637,591.81",1,"$5,000,000.00","$23,637,591.81"
6,2025-09-10,2025-09-24,"$6,927,191.81",14,"$-21,710,400.00","$28,637,591.81"
7,2025-09-11,2025-09-12,"$64,013,191.81",1,"$57,086,000.00","$6,927,191.81"
8,2025-09-12,2025-09-15,"$47,403,991.81",3,"$-16,609,200.00","$64,013,191.81"
9,2025-09-13,2025-09-15,"$47,403,991.81",2,"$-16,609,200.00","$64,013,191.81"
